# CGT-01 模型组装 教案

**课程名称：** 自建GPT训练流程 Part 1：模型组装——从零构建一个完整的 Transformer 模型

**预计总时长：** 100-110 分钟

**源文件：** `Custom_GPT_Training/01_Model_Assembly.ipynb`（共 33 个 Cell，Cell 0-32）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 00:00-10:00 | 开场、课程总览与环境准备 | Cell 0-3 | 10 分钟 |
| 10:00-22:00 | 四大核心技术概览 + RMSNorm | Cell 4-7 | 12 分钟 |
| 22:00-35:00 | SwiGLU 激活函数 | Cell 8-9 | 13 分钟 |
| 35:00-40:00 | 休息 + 回顾 | -- | 5 分钟 |
| 40:00-58:00 | RoPE 旋转位置编码 | Cell 10-11 | 18 分钟 |
| 58:00-68:00 | Pre-norm + 模型配置与创建 | Cell 12-15 | 10 分钟 |
| 68:00-73:00 | 休息 + 回顾 | -- | 5 分钟 |
| 73:00-83:00 | 分词器构建与测试 | Cell 16-18 | 10 分钟 |
| 83:00-95:00 | 前向传播测试 + 过拟合验证 | Cell 19-28 | 12 分钟 |
| 95:00-105:00 | 模型保存/加载 + 总结 | Cell 29-32 | 10 分钟 |

---

## 课前准备

- [ ] 确认 PyTorch 已安装（源 notebook 使用 PyTorch + CUDA）
- [ ] 确认 matplotlib、numpy 可用
- [ ] 确认中文字体设置正确（Microsoft YaHei / SimHei）
- [ ] 确认 `custom_gpt.py` 模块存在于 `Custom_GPT_Training/` 目录下
- [ ] 提前运行一遍全部 Cell，确认无报错（特别是 Cell 3 的环境导入和 Cell 14 的模型创建）
- [ ] 准备白板/画板用于画 Transformer Block 内部结构图
- [ ] 回顾 Ch3-Ch6（Self-Attention、Transformer Block、GPT 架构、Tokenizer 原理）确保能自如衔接

---

## 第一段：开场、课程总览与环境准备（Cell 0-3）

📍 浏览 Cell 0-1（课程概述 Markdown），运行 Cell 3（环境设置与模块导入）

⏱ 时间分配：10 分钟（课程定位 3 分钟 + 自建GPT vs HuggingFace对比 3 分钟 + Cell 3 运行与确认 2 分钟 + 训练流程预览 2 分钟）

🎯 本段目标
- 理解为什么要自建GPT而不是直接用 HuggingFace GPT-2
- 对 01-05 的完整训练链路有全局认知
- 确认环境就绪，模块导入成功

🗣 讲课话术

> 大家好，今天开始一个新的系列——自建GPT训练流程。这个系列一共五个 notebook，从模型组装、预训练、SFT 指令微调、DPO 偏好对齐，到最后的评估对比，完整走一遍大模型训练的全链路。
>
> 先说一个问题：为什么不直接用 HuggingFace 上的 GPT-2？Cell 0 有一张对比表。GPT-2 最小版本有 124M 参数，需要 GPU 训练，而且很多内部实现是黑盒的——你很难看清楚每一层到底在做什么。
>
> 我们自建的 GPT 只有大约 ~2.73M 参数，CPU 就能跑，架构完全透明——每一行代码你都能看到。更重要的是，这个模型会贯穿后面四个 notebook，你可以亲眼看到同一个模型从「随机初始化」到「能说人话」到「对齐人类偏好」的全过程。
>
> 架构方面，我们用的是现代 LLM 的标配：RMSNorm 归一化、SwiGLU 激活、RoPE 位置编码、Pre-norm 结构——和 LLaMA、Qwen 用的技术完全一样，只是规模缩小了。
>
> 现在跑一下 Cell 3 确认环境。这个 Cell 导入了 PyTorch、matplotlib，然后从 `custom_gpt.py` 模块导入了我们自建的所有组件：`CustomGPT`、`GPTConfig`、`SimpleTokenizer` 等。
>
> 看输出，应该显示 `使用设备: cuda`（如果有 GPU）或 `使用设备: cpu`。两个都能跑，GPU 会快一些。
>
> Cell 1 底部有前置知识清单和预计用时：阅读 30 分钟、代码 20 分钟、练习 20 分钟。今天我们会把这些内容都覆盖到，但节奏会更紧凑。

👀 输出要点
- Cell 3 应输出：`使用设备: cuda`（或 `使用设备: cpu`）
- 无报错信息，所有 import 成功

❓ 预判问题

Q: ~2.73M 参数够用吗？能生成有意义的文本吗？
A: 足够用于教学和演示。我们的词表只有 5000，模型维度 256，训练后在小规模中文语料上可以生成基本通顺的文本。目标不是做出 ChatGPT，而是让大家理解每一步的原理。

Q: custom_gpt.py 里有什么？
A: 包含完整的模型定义（CustomGPT）、配置类（GPTConfig）、分词器（SimpleTokenizer）、以及一些辅助函数。这个 notebook 会逐步讲解里面的核心技术，然后用这些现成组件来组装模型。

Q: 这个系列和前面 Ch5 GPT 组装有什么区别？
A: Ch5 用的是教学版简化 GPT（93万参数，可学习位置编码）。这个系列用现代技术（RoPE、RMSNorm、SwiGLU），更接近真实的 LLaMA/Qwen 架构，而且会走完训练全链路。

➡️ 转场

> 好，环境就绪。在组装模型之前，我们先理解它的四个核心组件。第一个是 RMSNorm——一种比 LayerNorm 更高效的归一化方法。

---

## 第二段：四大核心技术概览 + RMSNorm（Cell 4-7）

📍 浏览 Cell 4-6（四大技术总览 + 核心原理回顾 + RMSNorm 理论 Markdown），运行 Cell 7（RMSNorm 实现与验证）

⏱ 时间分配：12 分钟（四大技术总览 2 分钟 + RMSNorm 理论 4 分钟 + Cell 7 代码讲解与 TODO 补全 4 分钟 + 输出分析 2 分钟）

🎯 本段目标
- 对四大核心技术（RMSNorm、SwiGLU、RoPE、Pre-norm）建立全局认知
- 理解 RMSNorm 与 LayerNorm 的区别：去掉均值中心化，只做缩放归一化
- 完成 Cell 7 的三个 TODO：计算均方、计算 RMS、归一化
- 通过输出验证 RMS 归一化后接近 1.0

🗣 讲课话术

> Cell 4 有一张四大技术对比表。大家先扫一眼：RMSNorm 管归一化、SwiGLU 管激活函数、RoPE 管位置编码、Pre-norm 管归一化顺序。这四个技术加在一起，就是 LLaMA、Qwen、Mistral 这些现代大模型的标配。
>
> Cell 5 是核心原理回顾，快速过一下。Self-Attention 大家已经学过了。残差连接和 Pre-Norm 在 Ch4 Transformer Block 里也讲过。今天重点放在新的组件上。
>
> 先看 RMSNorm。大家还记得 LayerNorm 吗？它对每个样本的隐藏状态做两件事：减去均值（中心化）、除以标准差（缩放）。公式是 `(x - mean) / sqrt(var + eps) * gamma + beta`。
>
> RMSNorm 的核心发现是：**LayerNorm 的效果主要来自缩放，中心化贡献很小**。这是 2019 年 Biao Zhang 和 Rico Sennrich 的论文证明的。
>
> 所以 RMSNorm 直接把均值那一步去掉了，公式简化为 `x / RMS(x) * gamma`，其中 `RMS(x) = sqrt(mean(x^2) + eps)`。不需要算均值，不需要减均值，不需要 beta 偏置。计算量减少约 10-15%。
>
> 用一个比喻：LayerNorm 是先把所有人排成一排，找到平均身高，然后让每个人的身高减去平均值（中心化），再除以身高的标准差（缩放）。RMSNorm 说，不用找平均身高了，直接除以「均方根身高」就行了，效果差不多但省去了求平均值那一步。
>
> 来看 Cell 7 的代码。`MyRMSNorm` 类有三个 TODO。
>
> 大家看看能不能自己填。提示在代码注释里：Step 1 用 `torch.mean(x ** 2, dim=-1, keepdim=True)`；Step 2 用 `torch.sqrt(mean_sq + self.eps)`；Step 3 就是 `x / rms * self.weight`。
>
> 运行看输出。输入的 RMS 是 11.1036（因为我们故意造了均值为 5、方差很大的数据），输出的 RMS 是 1.0000——说明归一化成功了。
>
> 关键的对比在下面：LayerNorm 输出均值是 -0.000000（接近 0），而 RMSNorm 输出均值是 0.475157（不为 0）。这就是区别——RMSNorm 不做均值中心化，所以输出的均值不一定是零。但训练效果几乎一样好。

👀 输出要点
- Cell 7 输出：
  - `输入 RMS:  11.1036`
  - `输出 RMS:  1.0000  (应接近 1.0)` —— 验证归一化成功
  - `输出形状:  torch.Size([2, 5, 64])  (应与输入相同)`
  - LayerNorm 输出均值接近 0，RMSNorm 输出均值为 0.475157（不为 0）
  - 最后一行：`→ RMSNorm 不做均值中心化，这是与 LayerNorm 的关键区别！`

❓ 预判问题

Q: 为什么 RMSNorm 不需要 beta 偏置参数？
A: LayerNorm 有 gamma 和 beta 两个可学习参数，beta 用于中心化后的偏移。RMSNorm 不做中心化，自然也不需要 beta。只保留 gamma（缩放参数）。参数量减半。

Q: eps 的作用是什么？
A: 数值稳定性。如果输入全是零，`mean(x^2)` 就是零，`sqrt(0)` 会导致除以零。加个极小值 1e-5 避免这种情况。

Q: `keepdim=True` 为什么必须加？
A: 为了广播。不加 keepdim，均值的维度会被压缩，无法和原始张量做除法。加了 keepdim 保持维度一致，PyTorch 就能自动广播。

➡️ 转场

> RMSNorm 搞定了。第二个核心技术是 SwiGLU——一种门控激活函数，用来替代传统 FFN 里的 ReLU。

---

## 第三段：SwiGLU 激活函数（Cell 8-9）

📍 浏览 Cell 8（SwiGLU 理论 Markdown），运行 Cell 9（SwiGLU 实现 + 激活函数可视化）

⏱ 时间分配：13 分钟（传统FFN回顾 2 分钟 + GLU/SwiGLU 理论 4 分钟 + Cell 9 代码讲解与 TODO 3 分钟 + 可视化分析 2 分钟 + 参数量对比 2 分钟）

🎯 本段目标
- 理解门控机制的核心思想：用一个信号控制另一个信号的通过
- 掌握 SwiGLU 的三层结构（W1门控 + W3内容 + W2降维）
- 完成 Cell 9 的三个 TODO：门控信号、内容信号、输出
- 通过可视化理解 SiLU/Swish 与 ReLU/GELU 的区别

🗣 讲课话术

> Cell 8 先回顾了传统 FFN：两层线性变换，中间夹一个 ReLU。先升维到 4 倍（比如 256 -> 1024），再降维回来（1024 -> 256）。这个设计大家在 Ch4 就见过了。
>
> SwiGLU 的核心创新是引入了「门控机制」。用一个比喻来理解：传统 FFN 就像一条直通的水管，水（信息）直接流过去，中间被 ReLU 过滤一下。SwiGLU 加了一个「水龙头阀门」——有两条路径：
>
> 第一条是「内容路径」：x 乘以 W3，得到要传递的信息。
> 第二条是「门控路径」：x 乘以 W1，再过 SiLU 激活，得到一个 0 到 1 之间的阀门值。
> 然后两条路径逐元素相乘——阀门决定每个维度放多少信息通过。最后再乘以 W2 降维。
>
> 公式写出来就是：`FFN(x) = (SiLU(x @ W1) * (x @ W3)) @ W2`。注意和传统 FFN 最大的区别：SwiGLU 有三个线性层（W1、W2、W3），传统 FFN 只有两个（W1、W2）。
>
> SiLU 是什么？全称 Sigmoid Linear Unit，也叫 Swish。公式是 `x * sigmoid(x)`。它和 ReLU 的区别是：ReLU 在负数区域直接归零，SiLU 在负数区域有一个小的负值——更平滑，梯度不会突然消失。
>
> 来看 Cell 9 的代码。`MySwiGLU_FFN` 类有三个 TODO：
> - Step 1：`gate = F.silu(self.w1(x))` —— 门控信号
> - Step 2：`content = self.w3(x)` —— 内容信号
> - Step 3：`output = self.w2(gate * content)` —— 门控乘内容，降维
>
> 注意代码里 `bias=False`——现代 LLM 通常不用偏置，这也是和传统 FFN 的区别之一。
>
> 运行看输出。输入 [2, 10, 64]，输出也是 [2, 10, 64]，形状不变。参数量对比：标准 FFN 是 64*256*2 = 32,768，SwiGLU 是 64*256*3 = 49,152。多了 50% 的参数，但效果好——Noam Shazeer 2020 年的论文证明 SwiGLU 比 ReLU 的 PPL（困惑度）下降 2-5%。
>
> 下面的可视化图很直观：蓝色 ReLU 在零点有个尖角，负数全是零；绿色 GELU 比较平滑，负数区域有小的非零值；红色 SiLU/Swish 和 GELU 形状类似，在负数区域也有小负值。SiLU 的特点是它的「负值凹陷」比 GELU 稍微深一点。

👀 输出要点
- Cell 9 输出：
  - `输入: torch.Size([2, 10, 64])`
  - `输出: torch.Size([2, 10, 64])  (应与输入相同)`
  - `参数量: 49,152`
  - `对比标准 FFN: 32,768 (2 个 W)`
  - `SwiGLU FFN:  49,152 (3 个 W)`
- 一张激活函数对比图（ReLU 蓝色 / GELU 绿色 / SiLU 红色）
- 最后一行：`SiLU 特点: 负数区域有小的负值（与 GELU 类似），比 ReLU 更平滑`

❓ 预判问题

Q: SwiGLU 参数多了 50%，不是浪费吗？
A: 实际使用时会调整 d_ff 的大小。Cell 8 的表格提到，调整后参数量约相当：标准 FFN 用 `d_ff = 4 * d_model`，SwiGLU 用 `d_ff = 8/3 * d_model`，总参数量接近。

Q: SiLU 和 Swish 是同一个东西吗？
A: 是的。Swish 是 Google 提出的名字（Ramachandran et al., 2017），SiLU 是后来更通用的名字。PyTorch 里 `F.silu()` 就是 Swish。

Q: 门控机制和 LSTM 的门有什么关系？
A: 思想一样！LSTM 有输入门、遗忘门、输出门，都是用 sigmoid 输出 0-1 的值来控制信息流。SwiGLU 的门控也是类似的——用 SiLU 激活后的值来决定每个维度的信息通过量。

➡️ 转场

> 两个组件搞定了。先休息 5 分钟，回来看第三个核心技术——RoPE 旋转位置编码。

---

## 休息 + 回顾（第 35-40 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 我们自建的 GPT 用四大现代技术（RMSNorm、SwiGLU、RoPE、Pre-norm），和 LLaMA/Qwen 同源，参数量约 ~2.73M，CPU 可跑，完全透明。
2. RMSNorm 去掉了 LayerNorm 的均值中心化，只做 `x / RMS(x) * gamma`，计算量减少 10-15%。验证：输出 RMS = 1.0000，输出均值 != 0。
3. SwiGLU 用门控机制替代 ReLU：`(SiLU(xW1) * xW3) W2`。三个线性层（vs 传统两个），门控路径决定信息通过量，PPL 下降 2-5%。

**下一段预告：**

> 接下来看 RoPE——为什么现代大模型都不用加法式位置编码了，而是用「旋转」来注入位置信息？

---

## 第四段：RoPE 旋转位置编码（Cell 10-11）

📍 浏览 Cell 10（RoPE 理论 Markdown），运行 Cell 11（RoPE 实现 + 可视化）

⏱ 时间分配：18 分钟（位置编码动机 2 分钟 + RoPE 理论直觉 5 分钟 + Cell 11 代码讲解与 TODO 补全 6 分钟 + 可视化分析 3 分钟 + 与绝对位置编码对比 2 分钟）

🎯 本段目标
- 理解为什么 Attention 需要位置编码（排列不变性问题）
- 掌握 RoPE 的核心思想：不加位置信息，而是旋转 Q/K 使点积只依赖相对距离
- 完成 Cell 11 的 TODO：预计算频率、应用旋转
- 通过可视化理解「低维高频、高维低频」和「对角线模式=相对位置」

🗣 讲课话术

> Cell 10 开门见山提了一个问题：Self-Attention 是排列不变的。什么意思？就是把「狗咬人」的三个 token 打乱成「人咬狗」，Attention 的计算结果不会改变——因为 Attention 只看 Q 和 K 的点积，不管它们排在第几位。
>
> 但语言是有顺序的！「狗咬人」和「人咬狗」意思完全不同。所以必须想办法把位置信息注入进去。
>
> GPT-2 用的是最简单的方式——可学习的绝对位置嵌入：`h = TokenEmb(x) + PosEmb(position)`。问题是：训练时最大长度固定（比如 1024），超出就不行了；而且位置信息加在 Embedding 上，经过多层之后会被「冲淡」。
>
> RoPE 的思路完全不同。它不是把位置「加」到输入上，而是在计算 Q 和 K 的点积时，用旋转来编码位置。
>
> 我用一个钟表的比喻来解释。想象每个 token 手里拿着一根指针。位置 0 的指针指向 12 点方向，位置 1 旋转到 1 点，位置 2 旋转到 2 点......两个 token 做点积时，关键的是它们之间的「夹角」——这个夹角只取决于两者的相对距离，不取决于绝对位置。
>
> 数学上，RoPE 把 Q 和 K 的每对维度 (q_{2i}, q_{2i+1}) 做二维旋转。位置 m 旋转 m*theta 度。点积展开后：旋转后的 Q 点乘旋转后的 K = q^T * R(n-m) * k，只依赖于 n-m 这个相对距离。
>
> 还有一个重要的性质：旋转是正交变换，不改变向量的模长。这意味着 RoPE 不会影响 Q/K 的「大小」，只影响它们的「方向」。
>
> Cell 11 的代码分两部分。第一部分 `precompute_rope_frequencies` 预计算频率。关键公式 `inv_freq = 1 / (base^(2i/d))`——i 小（低维度）频率高，旋转快；i 大（高维度）频率低，旋转慢。然后对每个位置 m 算 `m * inv_freq` 得到旋转角度。
>
> 第二部分 `apply_rope` 对 Q 和 K 做旋转。公式是 `q' = q * cos + rotate_half(q) * sin`。`rotate_half` 把向量的前后两半交换并取负：[a, b] -> [-b, a]，这正是二维旋转矩阵的等价写法。
>
> TODO 比较直接：频率计算用 `torch.outer(positions, inv_freq)`，旋转用 `q * cos + rotate_half(q) * sin`，K 和 Q 一模一样。
>
> 运行看输出。关键验证：旋转前后 Q 的模长几乎不变（4.xxxx vs 4.xxxx），证明旋转是正交变换。
>
> 看可视化。左图是 cos(m*theta) 的热力图。底部（低维度）条纹密集——频率高，每个位置变化很大。顶部（高维度）条纹稀疏——频率低，变化很慢。这和正弦位置编码的「多频率钟表」思想一脉相承。
>
> 右图特别有意思：用统一的 Q 和 K（全是 1），应用 RoPE 后计算注意力分数矩阵。你可以看到明显的对角线模式——对角线上分数最高（位置相同，相对距离为 0），离对角线越远分数越低（相对距离越大）。这就是 RoPE 成功编码相对位置信息的证据。

👀 输出要点
- Cell 11 输出：
  - `Q 形状: torch.Size([1, 32, 16]) -> Q_rotated: torch.Size([1, 32, 16])`
  - `旋转前 Q 模长: X.XXXX`，`旋转后 Q 模长: X.XXXX`（两者接近相等）
  - `→ 模长几乎不变（旋转是正交变换！）`
- 两张并排图：
  - 左图：`RoPE cos(m*theta)` 热力图，低维度条纹密集，高维度条纹稀疏
  - 右图：`RoPE 注意力分数` 热力图，呈现对角线模式
- 最后一行：`观察: 注意力分数呈现明显的对角线模式，说明 RoPE 成功编码了相对位置信息`

❓ 预判问题

Q: `rotate_half` 为什么是 [-b, a] 而不是其他排列？
A: 这对应二维旋转矩阵的结构。旋转矩阵 R(theta) = [[cos, -sin], [sin, cos]]，作用在 [a, b] 上得到 [a*cos - b*sin, a*sin + b*cos]。拆开后就是 `[a,b] * cos + [-b,a] * sin`。所以 rotate_half 就是旋转矩阵公式的第二项。

Q: RoPE 能处理超出训练长度的序列吗？
A: 原始 RoPE 在超出训练长度后效果会退化，但配合 NTK-aware 缩放、YaRN 等技术可以外推到更长长度。这比绝对位置编码的外推能力强很多。

Q: 为什么 RoPE 只作用在 Q 和 K 上，不作用在 V 上？
A: 因为位置信息通过 Q*K 的点积影响注意力分数。V 是被加权求和的「内容」，不需要位置旋转。RoPE 的设计目标就是让注意力分数天然包含相对位置信息。

Q: 频率基数 10000 有什么特殊意义？
A: 控制最长波长。最低频率维度的旋转周期约为 10000 个位置，足以覆盖大多数序列。这是经验值，有些模型会用不同的 base（如 Qwen 用 1000000）。

➡️ 转场

> 三大核心技术讲完了。最后一个 Pre-norm 比较简单——就是归一化放在 Attention/FFN 之前而不是之后。然后我们就正式开始组装模型。

---

## 第五段：Pre-norm + 模型配置与创建（Cell 12-15）

📍 浏览 Cell 12（Pre-norm 理论 + Transformer Block 结构图 Markdown），运行 Cell 13（GPTConfig 配置打印），运行 Cell 14（模型创建 + 摘要），运行 Cell 15（架构可视化图）

⏱ 时间分配：10 分钟（Pre-norm 理论 2 分钟 + Cell 13 配置讲解 2 分钟 + Cell 14 模型创建 3 分钟 + Cell 15 架构图 3 分钟）

🎯 本段目标
- 理解 Pre-norm 比 Post-norm 更稳定的原因（残差连接的梯度不经过归一化）
- 完整理解一个 Transformer Block 的数据流
- 掌握模型的具体配置参数和参数量
- 理解配置中的参数量预估和实际参数量的差异

🗣 讲课话术

> Cell 12 讲 Pre-norm vs Post-norm。原始 Transformer 是 Post-norm：先做 Attention，再加残差，最后 LayerNorm。现代大模型都换成了 Pre-norm：先 RMSNorm，再做 Attention，最后加残差。
>
> 为什么 Pre-norm 更稳定？关键在反向传播。Post-norm 的梯度要穿过 LayerNorm 这个非线性变换，可能不稳定。Pre-norm 里，残差连接提供了一条梯度的「直通道」——梯度可以不经过归一化直接流回去，LayerNorm 只影响 Attention 那个支路。这让深层模型训练更稳定。
>
> Cell 12 底部有一个 ASCII 图，展示了完整的 Transformer Block 结构：输入 -> RMSNorm -> Multi-Head Attention (with RoPE) -> Dropout -> 残差连接 -> RMSNorm -> SwiGLU FFN -> Dropout -> 残差连接 -> 输出。每个组件大家都已经理解了。
>
> 好，现在正式开始组装。Cell 13 打印了 GPTConfig 的默认配置。注意这些关键数字：
> - vocab_size = 5000：词表 5000 个 token
> - d_model = 192：模型维度
> - n_heads = 4：4 个注意力头
> - n_layers = 4：4 层 Transformer Block
> - d_ff = 512：FFN 维度，约为 d_model 的 2.67 倍
> - max_seq_len = 128：最大序列长度
> - dropout = 0.1：标准的 dropout 比例
>
> 预估参数量约 ~2.73M。然后 Cell 14 创建模型并打印摘要——实际参数量是 2.73M。为什么比预估多了？因为预估可能只算了主要组件（Embedding + Transformer Blocks），实际还包括 LM Head、最终的 RMSNorm 等。
>
> Cell 15 的架构可视化图从上到下展示了完整的数据流：输入 Token IDs -> Token Embedding -> 4 个 Transformer Block（每个包含 RMSNorm + 多头注意力 + RoPE 和 RMSNorm + SwiGLU FFN）-> 最终 RMSNorm -> LM Head。颜色编码很清晰——蓝色是嵌入层、黄色是注意力、绿色是 FFN、粉色是归一化、紫色是输出头。

👀 输出要点
- Cell 13 输出配置参数表，关键数值：
  - `vocab_size: 5000`, `d_model: 384`, `n_heads: 6`, `n_layers: 6`, `d_ff: 1536`
  - `预估参数量: ~2.73M`
- Cell 14 输出模型摘要：
  - `Total Parameters: 2.73M`
  - `Trainable Parameters: ~2,730,000`
- Cell 15 输出一张彩色架构可视化图（11x13），展示 4 个 Transformer Block 的堆叠

❓ 预判问题

Q: 为什么 d_ff 是 d_model 的 4 倍？
A: 这是原始 Transformer 论文的设计。FFN 先升维再降维，升维 4 倍给了模型更大的中间表示空间，提升非线性表达能力。4 倍是经验值，现代模型有时用 8/3 倍（配合 SwiGLU 三层结构）。

Q: 2.73M 参数是怎么算出来的？
A: 预估值可能只计算了主要组件。实际参数还包括 LM Head（如果不共享权重的话，5000 * 192 = 0.96M）、每层的 RMSNorm 参数、最终的 RMSNorm 等。

Q: 这个模型和真正的 LLaMA 有什么区别？
A: 核心架构完全一致（RMSNorm + SwiGLU + RoPE + Pre-norm），但规模差很多。LLaMA-7B 有 32 层、d_model=4096、32 个注意力头。我们的模型是 4 层、d_model=192、4 个头，大约是 LLaMA 的千分之二。

➡️ 转场

> 模型架构组装完了。但模型只能处理数字——它不认识中文字。我们还需要一个分词器来做文本和数字之间的转换。先休息 5 分钟。

---

## 休息 + 回顾（第 68-73 分钟）

⏱ 时间分配：5 分钟

**三句话回顾：**

1. RoPE 不是加位置信息，而是在计算 Q*K 点积时旋转向量，使注意力分数天然只依赖相对距离 m-n。旋转是正交变换，不改变向量模长。可视化中对角线模式证实了相对位置编码的有效性。
2. Pre-norm 把归一化放在 Attention/FFN 之前，残差连接提供梯度直通道，比 Post-norm 训练更稳定。完整的 Transformer Block：RMSNorm -> Attention(RoPE) -> 残差 -> RMSNorm -> SwiGLU -> 残差。
3. 模型配置：vocab_size=5000, d_model=192, n_heads=4, n_layers=4, d_ff=512。实际参数量 2.73M，CPU 可跑。

**下一段预告：**

> 接下来构建分词器，让模型能处理中文文本。然后做前向传播测试和过拟合验证，确认模型能正常工作。

---

## 第六段：分词器构建与测试（Cell 16-18）

📍 浏览 Cell 16（分词器 Markdown），运行 Cell 17（构建词表），运行 Cell 18（编码/解码测试）

⏱ 时间分配：10 分钟（分词器概念 2 分钟 + Cell 17 词表构建 4 分钟 + Cell 18 编码解码测试 4 分钟）

🎯 本段目标
- 理解字符级分词器的工作原理
- 掌握特殊 token（PAD、BOS、EOS、UNK）的作用
- 验证编码-解码的完整性
- 理解 padding 的目的

🗣 讲课话术

> Cell 16 说我们要创建一个简单的字符/词级分词器。为什么不用 BPE？因为我们的模型很小，词表只有 5000，用字符级分词足够了。而且字符级分词对中文天然友好——每个汉字就是一个 token。
>
> Cell 17 先准备了 10 句中文示例文本，重复 100 遍来获得足够的频率统计。然后用 `SimpleTokenizer` 构建词表。
>
> 看输出：`词表大小: 127`。虽然我们设置了 vocab_size=5000，但实际只有 127 个不同的字符出现。这很正常——10 句话的字符多样性有限。
>
> 词表的前几项是特殊 token：PAD(0)、BOS(1)、EOS(2)、UNK(3)。这四个特殊 token 的作用：
> - PAD：填充，让不同长度的句子对齐到相同长度（batch 训练需要）
> - BOS（Begin of Sequence）：序列开始标记，告诉模型「新句子开始了」
> - EOS（End of Sequence）：序列结束标记，告诉模型「句子到此为止」
> - UNK：未知 token，遇到词表里没有的字符时使用
>
> 然后看 Cell 18 的编码解码测试。原文是「深度学习是人工智能的核心技术。」，编码后变成 `[1, 57, 11, 9, 10, 7, 50, 51, 52, 20, 5, 3, 3, 102, 103, 4, 2]`。
>
> 注意开头的 1 是 BOS，结尾的 2 是 EOS。中间有两个 3（UNK）——这说明「核」和「心」这两个字不在词表里！解码后变成「深度学习是人工智能的技术。」——「核心」丢失了。
>
> 这就是小词表的局限。真实的大模型用 BPE 算法、几万个 token 的词表来覆盖更多字符和子词。
>
> 最后 padding 的例子：把句子填充到长度 30，后面补了很多 0（PAD token）。这样不同长度的句子可以放在同一个 batch 里一起训练。

👀 输出要点
- Cell 17 输出：
  - `Vocabulary built: 127 tokens`
  - `词表大小: 127`
  - 部分词表内容：`'<PAD>' -> 0`, `'<BOS>' -> 1`, `'<EOS>' -> 2`, `'<UNK>' -> 3`, `'。' -> 4`, `'的' -> 5` 等
- Cell 18 输出：
  - 编码：`[1, 57, 11, 9, 10, 7, 50, 51, 52, 20, 5, 3, 3, 102, 103, 4, 2]`
  - 解码：`深度学习是人工智能的技术。`（注意「核心」变成了 UNK）
  - 填充到长度30：末尾补大量 0

❓ 预判问题

Q: 为什么「核」和「心」是 UNK？
A: 因为这两个字没有出现在训练文本的 10 句话中（或出现频率低于 min_freq）。字符级分词器的词表完全取决于训练文本的内容。

Q: BOS 和 EOS 在训练时有什么用？
A: BOS 让模型知道序列起点，生成时从 BOS 开始生成。EOS 让模型学会在合适的时候停止生成。没有 EOS 的话，模型会一直生成下去，不知道什么时候该停。

Q: 字符级分词和 BPE 的区别？
A: 字符级：每个字符一个 token，词表小但序列长。BPE：把高频字符组合合并成子词，词表大但序列短。比如「学习」在 BPE 里可能是一个 token，字符级里是两个。真实大模型几乎都用 BPE 或 SentencePiece。

➡️ 转场

> 分词器有了，模型有了，现在来测试它们能不能正常工作。先测前向传播，再做一个过拟合测试——让模型在一句话上反复学习，看 loss 能不能降下来。

---

## 第七段：前向传播测试 + 过拟合验证（Cell 19-28）

📍 运行 Cell 20（前向传播测试），Cell 21（训练模式 + loss），Cell 22（Attention Mask 测试），Cell 23（文本生成测试），Cell 25-26（过拟合数据准备 + 训练），Cell 27（loss 曲线），Cell 28（训练后生成测试）

⏱ 时间分配：12 分钟（Cell 20 前向传播 2 分钟 + Cell 21 loss 计算 2 分钟 + Cell 22 Mask 1 分钟 + Cell 23 未训练生成 1 分钟 + Cell 25-26 过拟合训练 3 分钟 + Cell 27-28 验证 3 分钟）

🎯 本段目标
- 验证模型前向传播输出形状正确：[batch, seq_len, vocab_size]
- 理解初始 loss 应接近 ln(vocab_size)
- 理解 Attention Mask 如何处理 padding
- 通过过拟合测试验证模型能学习（loss 从 8.67 降到 0.02）
- 对比训练前后的生成质量

🗣 讲课话术

> 现在进入验证环节。Cell 20 做前向传播测试。随机生成 batch_size=4、seq_len=32 的输入 ID，喂给模型。
>
> 输出 logits 形状是 [4, 32, 5000]——batch 4、序列长度 32、词表 5000。每个位置输出 5000 个分数，代表下一个 token 是词表中每个词的「打分」。这和我们 Ch5 讲的完全一致。
>
> Cell 21 测试训练模式——传入 labels 就会自动计算 loss。初始 loss 是 8.5602，而 ln(5000) = 8.5172。为什么接近？因为模型还没训练，参数是随机的，对 5000 个词的预测基本是均匀分布。均匀分布的交叉熵就是 ln(5000)。这是一个很好的 sanity check——如果初始 loss 远离这个值，说明模型有 bug。
>
> Cell 22 测试 Attention Mask。两个句子长度不同——「深度学习」只有 6 个 token（含 BOS/EOS），「机器学习是人工智能」有 11 个。都 padding 到 20。Attention Mask 是一个 0/1 矩阵，真实 token 的位置是 1，padding 位置是 0。这样模型在做 Attention 时就会忽略 padding 位置。
>
> Cell 23 测试未训练的生成。Prompt 是「深度学习」，生成 20 个新 token。看输出——生成的 ID 都是几千的大数字，解码后完全是乱码。这很正常，模型还没训练过，输出自然是随机的。
>
> 现在做关键的过拟合测试。Cell 25 准备数据：「深度学习是人工智能的核心技术。」这一句话。输入是整句话去掉最后一个 token，标签是去掉第一个 token——也就是经典的 next-token prediction 对齐方式。
>
> Cell 26 训练 100 步，学习率 1e-3。看输出：Step 0 的 loss 是 8.6669（接近 ln(vocab_size)），Step 20 就降到了 0.0936，最终降到 0.0177。从 8.67 到 0.02，下降了 400 多倍！
>
> 这说明什么？说明模型的梯度流通正常，优化器能有效更新参数，模型有能力记住信息。这是模型正确性的重要验证。
>
> Cell 27 画了 loss 曲线图，可以看到 loss 在前 20 步急剧下降，然后趋于平稳。
>
> Cell 28 用训练后的模型生成。Prompt 还是「深度学习」，期望输出是「深度学习是人工智能的核心技术。」。实际生成是「深度学习人智智的术工智的术」——不完美，但比训练前的纯乱码好多了。记住我们只在一句话上训练了 100 步，而且 temperature=0.5 引入了随机性。

👀 输出要点
- Cell 20：`输出logits形状: torch.Size([4, 32, 5000])`
- Cell 21：`Loss: 8.5602`，`初始loss期望值 ~ ln(vocab_size) = 8.5172`
- Cell 22：两个 Attention Mask 向量，第一个有 6 个 1，第二个有 11 个 1
- Cell 23：生成乱码（`深度学习` 后面跟随机字符），提示「模型尚未训练」
- Cell 26：loss 从 8.6669 下降到 0.0177
  - Step 0: 8.6669
  - Step 20: 0.0936
  - Step 40: 0.1071
  - Step 60: 0.0127
  - Step 80: 0.0349
  - 最终: 0.0177
- Cell 27：loss 曲线图，前 20 步急剧下降
- Cell 28：生成 `深度学习人智智的术工智的术`，期望 `深度学习是人工智能的核心技术。`

❓ 预判问题

Q: 为什么初始 loss 接近 ln(vocab_size)？
A: 未训练的模型对 5000 个词的预测接近均匀分布。均匀分布的概率是 1/5000，交叉熵 = -ln(1/5000) = ln(5000) = 8.5172。这是一个经典的 sanity check。

Q: 过拟合测试为什么重要？
A: 如果模型在一句话上都无法过拟合（loss 降不下来），说明模型架构、梯度计算、优化器配置中有 bug。能过拟合不代表模型能泛化，但不能过拟合一定是有问题。这是调试深度学习模型的第一步。

Q: Cell 28 生成的文本为什么不完美？
A: 几个原因：只训练了 100 步；temperature=0.5 引入了采样随机性；「核心」两个字在词表里是 UNK，模型根本不认识这两个字。

Q: Step 40 的 loss (0.1071) 比 Step 20 (0.0936) 还高？
A: 这是正常的。训练 loss 不是严格单调下降的，特别是在接近收敛时会有小幅波动。整体趋势是下降的就没问题。

➡️ 转场

> 模型验证通过了——前向传播正常、loss 能下降、生成功能正常。最后一步：把模型和分词器保存下来，供后续 notebook 使用。

---

## 第八段：模型保存/加载 + 总结（Cell 29-32）

📍 运行 Cell 30（保存模型和分词器），Cell 31（加载验证），浏览 Cell 32（总结 Markdown）

⏱ 时间分配：10 分钟（Cell 30 保存 2 分钟 + Cell 31 加载验证 2 分钟 + Cell 32 总结 3 分钟 + 下一步预告 3 分钟）

🎯 本段目标
- 理解模型保存/加载的工程实践
- 验证加载后参数量和词表一致
- 总结本节全部内容
- 预告下一步预训练

🗣 讲课话术

> Cell 30 做了两件事：第一，创建一个全新的、未训练的模型（不是我们刚才过拟合的那个），保存到 `models/custom_gpt/base_model/`。第二，保存分词器到 `models/custom_gpt/tokenizer.pkl`。
>
> 为什么保存未训练的模型？因为后续的预训练 notebook (02) 需要从零开始训练。如果保存了过拟合的版本，那就不是真正的「预训练」了。
>
> Cell 31 验证加载功能。加载模型后参数量是 2.73M，和创建时一致。分词器词表大小 127，也一致。说明保存和加载都正确。
>
> Cell 32 总结了本 notebook 的三大成果：
>
> 第一，我们自建了一个 GPT 模型，用了四大现代技术——RoPE 位置编码、RMSNorm 归一化、SwiGLU 激活函数、Pre-norm 架构。
>
> 第二，我们构建了一个字符级分词器，支持 PAD、BOS、EOS、UNK 四种特殊 token，支持保存和加载。
>
> 第三，我们做了完整的功能验证——前向传播、loss 计算、过拟合测试、文本生成。
>
> 下一步是 02_Pretraining.ipynb——在文本语料上做 next-token prediction 预训练，让模型从「随机输出」变成「能说人话」。这是大模型训练的第一阶段，也是最重要的一步。
>
> 最后给大家三个面试高频题做课后思考：
>
> 第一，RMSNorm 和 LayerNorm 的区别是什么？——去掉均值中心化，只做缩放。
> 第二，为什么 RoPE 的点积只依赖相对位置？——旋转矩阵的正交性：R(m)^T * R(n) = R(n-m)。
> 第三，过拟合测试的意义是什么？——验证模型架构和梯度流通正确，是调试的第一步。

👀 输出要点
- Cell 30 输出：
  - `Model saved to ...\models\custom_gpt\base_model`
  - 保存路径信息
- Cell 31 输出：
  - `Model loaded from ...\models\custom_gpt\base_model`
  - `成功加载模型和分词器!`
  - `模型参数: 2.73M`
  - `分词器词表大小: 127`
- Cell 32：总结 Markdown，列出三大成果和下一步预告

❓ 预判问题

Q: 预训练需要多少数据？
A: 我们这个小模型用几 MB 的中文文本就够了。真实的大模型需要几 TB 的数据。02 notebook 会详细讲数据准备。

Q: 预训练后就能对话了吗？
A: 不能。预训练只让模型学会「续写」——给一段文本，接着写下去。要能对话还需要 SFT（03 notebook）教它遵循指令，然后 DPO（04 notebook）对齐人类偏好。

Q: save_pretrained 和 torch.save 有什么区别？
A: save_pretrained 是 custom_gpt 模块提供的高级接口，会同时保存模型权重和配置信息，加载时可以自动重建模型。torch.save 只保存权重，需要手动重建模型结构。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，课程总览，环境准备 | 0-3 |
| 10 | 四大核心技术概览 + RMSNorm 理论与实现 | 4-7 |
| 22 | SwiGLU 理论与实现 + 激活函数可视化 | 8-9 |
| 35 | **休息** | -- |
| 40 | RoPE 理论与实现 + 可视化 | 10-11 |
| 58 | Pre-norm + 模型配置与创建 + 架构图 | 12-15 |
| 68 | **休息** | -- |
| 73 | 分词器构建与测试 | 16-18 |
| 83 | 前向传播 + 过拟合训练 + 生成测试 | 19-28 |
| 95 | 模型保存/加载 + 总结 + 下一步预告 | 29-32 |
| 105 | 结束 | -- |

---

## 附录 B：关键数据快速参考

### 核心公式

| 公式 | 表达式 | 说明 |
|:---|:---|:---|
| RMSNorm | RMSNorm(x) = x / sqrt(mean(x^2) + eps) * gamma | 只做缩放，不做中心化 |
| SwiGLU | FFN(x) = (SiLU(xW1) * xW3) W2 | 三层线性变换 + 门控 |
| SiLU/Swish | SiLU(x) = x * sigmoid(x) | 平滑激活函数 |
| RoPE 频率 | theta_i = 1/10000^{2i/d} | 低维高频，高维低频 |
| RoPE 旋转 | q' = q * cos(m*theta) + rotate_half(q) * sin(m*theta) | 只依赖相对位置 m-n |
| 初始 loss | -ln(1/vocab_size) = ln(vocab_size) | 均匀分布的交叉熵 |

### 模型配置速查

| 参数 | 值 | 说明 |
|:---|:---|:---|
| vocab_size | 5,000 | 词表大小 |
| d_model | 192 | 模型隐藏维度 |
| n_heads | 4 | 注意力头数 |
| n_layers | 4 | Transformer Block 层数 |
| d_ff | 512 | FFN 隐藏维度（~2.67 * d_model） |
| max_seq_len | 128 | 最大序列长度 |
| dropout | 0.1 | Dropout 比例 |
| 实际参数量 | 2.73M | ~2,730,000 |

### 关键数值

| 数值 | 来源 | 含义 |
|:---|:---|:---|
| RMS 输出 = 1.0000 | Cell 7 | RMSNorm 归一化验证 |
| RMSNorm 均值 = 0.475157 | Cell 7 | 不做中心化的证据 |
| SwiGLU 参数 49,152 vs FFN 32,768 | Cell 9 | 三层 vs 两层的参数量差异 |
| 初始 loss = 8.5602 | Cell 21 | 接近 ln(5000)=8.5172 |
| 过拟合 loss: 8.67 -> 0.02 | Cell 26 | 100 步训练验证 |
| 词表实际大小 = 127 | Cell 17 | 10 句话的字符多样性 |
| 编码含 2 个 UNK (ID=3) | Cell 18 | 「核心」不在词表中 |

### 张量维度速查

| 变量 | 形状 | 来源 |
|:---|:---|:---|
| RMSNorm 输入/输出 | [2, 5, 64] | Cell 7 |
| SwiGLU FFN 输入/输出 | [2, 10, 64] | Cell 9 |
| RoPE Q/K | [1, 32, 16] | Cell 11 |
| RoPE cos/sin 缓存 | [max_seq_len, d_model] | Cell 11 |
| 模型输入 input_ids | [batch, seq_len] | Cell 20: [4, 32] |
| 模型输出 logits | [batch, seq_len, vocab_size] | Cell 20: [4, 32, 5000] |
| Attention Mask | [batch, seq_len] | Cell 22: [2, 20] |

---

## 附录 C：应急预案

### 场景 1：环境或模块导入失败

**症状：** Cell 3 报错 `ModuleNotFoundError: No module named 'custom_gpt'`

**应对：**
1. 确认当前工作目录是 `Custom_GPT_Training/` 或项目根目录
2. 确认 `custom_gpt.py` 文件存在于 `Custom_GPT_Training/` 下
3. 手动添加路径：`sys.path.insert(0, 'path/to/Custom_GPT_Training')`
4. 如果是 PyTorch 未安装：`pip install torch`

### 场景 2：中文字体不显示

**症状：** Cell 15 架构可视化图中文显示为方框

**应对：**
1. Cell 3 已配置 `plt.rcParams["font.sans-serif"]` 备选字体列表
2. 尝试：`plt.rcParams['font.sans-serif'] = ['SimHei']`
3. 实在不行用 `['DejaVu Sans']`（牺牲中文，保证图能看），口头补充中文标签含义

### 场景 3：Cell 7/9/11 的 TODO 学生卡住

**应对：**
- Cell 7 RMSNorm：三个 TODO 都有提示注释，给 2 分钟后直接展示。关键是理解 `keepdim=True`
- Cell 9 SwiGLU：三个 TODO 对应门控/内容/输出三步，提示「看公式 FFN(x) = (SiLU(xW1) * xW3) W2」
- Cell 11 RoPE：TODO 最多，但 `apply_rope` 的 K 和 Q 完全对称。提示「K 的公式和 Q 一模一样」
- 源 notebook 中 TODO 后面已有参考答案（带注释），必要时直接展示

### 场景 4：过拟合测试 loss 不下降

**症状：** Cell 26 的 loss 始终在 8.5 附近不下降

**应对：**
1. 检查模型是否在 `model.train()` 模式
2. 检查 `optimizer.zero_grad()` 是否在 `loss.backward()` 之前
3. 检查学习率是否合理（1e-3 应该足够）
4. 检查 labels 是否正确右移了一位
5. 重启 kernel，重新运行所有 Cell

### 场景 5：GPU 内存不足

**应对：**
1. 模型只有 16M 参数，通常不会 OOM
2. 如果确实 OOM，改用 CPU：在 Cell 3 中设置 `device = 'cpu'`
3. 或者减小 batch_size（Cell 20 的 batch_size=4 可以改为 2）

### 场景 6：时间不够

**可跳过的内容（按优先级）：**
1. Cell 22 Attention Mask 测试（口头一句话说明 mask 的作用）- 省 1 分钟
2. Cell 15 架构可视化图（口头描述数据流即可）- 省 3 分钟
3. Cell 11 RoPE 可视化分析（只看代码输出，不详细分析图）- 省 2 分钟
4. Cell 5 核心原理回顾 Markdown（学生已有前置知识）- 省 1 分钟

**不可跳过的核心：**
- Cell 7：RMSNorm 实现与验证（第一个核心技术）
- Cell 9：SwiGLU 实现与可视化（第二个核心技术）
- Cell 11：RoPE 实现（第三个核心技术，至少看代码和输出）
- Cell 13-14：模型配置与创建（本节标题就是「模型组装」）
- Cell 26：过拟合测试（验证模型正确性）
- Cell 30-31：保存加载（后续 notebook 依赖）

### 场景 7：学生提出超纲问题

**应对：**
- KV Cache：「生成时缓存历史 K/V，避免重复计算。后续 notebook 会详细讲」
- FlashAttention：「分块计算 Attention，减少内存访问。我们这个小模型不需要，但生产环境必用」
- 量化/LoRA：「这些是参数高效微调技术，在 SFT/DPO 阶段用到。系列后续会涉及」
- 记录在白板上，不展开以免偏离主线